### Setup

In [1]:
# Import libraries
import pandas as pd
import json
from openai import OpenAI
from google.cloud import bigquery
from google.cloud.exceptions import BadRequest

from dotenv import load_dotenv
import os

In [ ]:
load_dotenv(override= True)

# BigQuery setup
client = bigquery.Client(project= os.getenv('BIGQUERY_PROJECT_ID'))
max_gb=30 # Query GB limit

# OpenAI setup
api_key = os.getenv('OPENAI_API_KEY')
base_url = '<https://api.openai.com/v1>'
default_headers = {'Authorization': f'Bearer {api_key}'}
model_name = 'gpt-5-2025-08-07'

### Pre-prompt the Agent

In [ ]:
## Set up system prompt
system_prompt = """Your role is to translate requests to SQL queries for BigQuery.

IMPORTANT: You have access to tools to help you.

IMPORTANT BEHAVIOR RULES:
Always explain with one short sentence what you're going to do before using any tools
Don't just call tools without explanation."""

In [4]:
# User rules for custom agent behavior (optional)
user_rules = 'Use USING for joins - when possible. Put SQL keywords in uppercase.'

### Define Tools

In [5]:
def execute_query(query):
    
    print('Query:', query)
    
    # Step 1: Dry run to validate query and check cost (skip if too costly > max_gb)
    try:
        
        print('Performing dry run...')
        job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
        dry_run_job = client.query(query, job_config=job_config)        
        # Check if query is too expensive
        gb_processed = (dry_run_job.total_bytes_processed) / (1024 ** 3) # Calculate cost in bytes and convert to GB
        if gb_processed > max_gb:
            return {
                'success': False,
                'error': f'Query cost is {gb_processed:.2f} GB which is too costly (max: {max_gb} GB)',
                'data': None
            }
        print(f'Query cost: {gb_processed:.2f} GB')
        print('Dry run successful, proceeding with execution...')

    except Exception as e:
        print('Query failed, check the query syntax')
        return {
            'success': False,
            'error': f'Query validation failed: {str(e)}',
            'data': None,
        }
    
    # Step 2: Execute the actual query
    try:
        print('Executing query...')
        job = client.query(query)
        results = job.result()
        
        # Convert results to list of dictionaries for easier handling
        data = []
        for row in results:
            data.append(dict(row))
        
        print(f'Query executed successfully! Returned {len(data)} rows')
        # Print the preview in a dataframe
        df = pd.DataFrame(data)
        display(df)
        
        return {
            'success': True,
            'error': None,
            'data': data
        }
        
    except Exception as e:
        return {
            'success': False,
            'error': f'Query execution failed: {str(e)}',
            'data': None
        }

In [6]:
# Describe function for OpenAI
openai_tools = [
    {
    "type": "function",
    "function": {
        "name": "execute_query",
        "description": "Execute a SQL query on BigQuery. Returns the query results.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "BigQuery SQL query."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        },
        "strict": True
    }
    }
]

# Simple tool description - convert tool dict to string
tools_description = f"Available tools:\\n{json.dumps(openai_tools, indent=2)}"

In [7]:
# Tool registry (links tool names to functions)
TOOL_REGISTRY = {
    "execute_query": execute_query,
}

### Agentic Loop

In [8]:
def run_agent(user_prompt, context='', privacy_mode=True):
    
    # 0. Compile the system prompt + user prompt to the AI agent (Concatenate all prompts)
    messages = [
        {"role": "system", "content": system_prompt + "\\n" + tools_description},
        {"role": "user", "content": user_rules},
        {"role": "user", "content": user_prompt + "\\n" + context}
    ]

    llm_client = OpenAI(api_key=api_key, base_url=base_url, default_headers=default_headers)
    running = True

    while running:

        # 1. Send compiled prompt and get the first response from the agent
        response = llm_client.chat.completions.create(
            messages=messages,
            model=model_name,
            tools=openai_tools,
            parallel_tool_calls=False,
            stream=False,
        )
        
        message = response.choices[0].message # Get message (potentially has tool execution intent)
        print("Assistant:", message.content)
    
        # 2. Get the list of tools to execute; add assistant + tools messages to the stream of messages with the agent
        tool_calls = []
        if message.tool_calls:
            
            # Get list of tools to call
            for tool_call in message.tool_calls: 
                tool_calls.append({
                    "name": tool_call.function.name,
                    "args": tool_call.function.arguments,
                    "id": tool_call.id
                })
            
            # Add tool calls to messages
            messages.append({
                "role": "assistant",
                "content": message.content,
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments
                        }
                    } for tc in message.tool_calls
                ]
            })
        
        else:
            # Add assistant message to messages
            messages.append({
                "role": "assistant",
                "content" : message.contet
            })
        
        # 3. Execute the tools
        for tool_call in tool_calls:
            try:
                print(f"Calling: {tool_call['name']}")
                result = TOOL_REGISTRY[tool_call["name"]](**json.loads(tool_call["args"]))

                # Don't send data content back to the user unless excplicity allowed
                if privacy_mode:
                    result = 'Results hidden due to privacy mode'

                # Send tool results back to LLM
                messages.append({
                    "role": "tool",
                    "content": json.dumps(result),
                    "tool_call_id": tool_call["id"]
                })
            
            except Exception as e:
                print(f"Error calling tool {tool_call['name']}: {e}")
                messages.append({
                    "role": "tool",
                    "content": f"Error: {str(e)}",
                    "tool_call_id": tool_call["id"]
                })
        
        # 4. Running check (to stop the loop if no tools are executed)
        running = len(tool_calls) > 0

### Test Output